# Pruning

Structured pruning of Gemma-3 270M: MLP channels, attention heads, and whole blocks.

In [1]:
import torch
import torch_pruning as tp

from huggingface_hub import login
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import os
import dotenv
from pathlib import Path

dotenv.load_dotenv()

True

## Load XL-SUM dataset for validation

We need to validate pruned model to get the best score. We do not need all the dataset, there is a need to take at least 100 samples.

In [2]:
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

validation_dataset = load_dataset(
    "parquet",
    data_files={
        "validation": str(PROJECT_ROOT / "data" / "raw" / "xlsum" / "ukrainian" / "validation" / "0000.parquet")
    },
    split="validation",
)

In [3]:
print(validation_dataset[0])

{'id': 'news-41032282', 'url': 'https://www.bbc.com/ukrainian/news-41032282', 'title': 'Саудівська поліція відпустила підлітка, який танцював "макарену"', 'summary': 'Саудівська поліція відпустила 14-річного підлітка, якого затримали після публікації в інтернеті \u200eвідео, на якому юнак танцює "макарену" на дорозі. Йому не висунули жодних звинувачень.\u200e', 'text': 'Міністерство внутрішніх справ країни заявило, що юнак підписав письмову обіцянку того, що більше "не буде залучений до поведінки, яка може поставити під загрозу його життя та життя інших".\u200e У поліції не назвали його ім\'я, проте зазначили, що він є громадянином іншої арабської країни.\u200e \u200e45-секундний ролик був знятий минулого року. На ньому юнак перешкоджає рухові автомобілів, \u200eтанцюючи в навушниках на п\'ятисмуговій дорозі у саудівському місті Джидда.\u200e Юнака затримали у цей вівторок у місті Мекка - після того, як ролик раптово набув шаленої \u200eпопулярності у соцмережах. Його допитали з привод

In [4]:
val_data = validation_dataset.shuffle(seed=42).select(range(100))["text"]

## Load Gemma-3 270M

In [5]:
login(token=os.getenv("HF_TOKEN"), add_to_git_credential=False)
device = "cpu"

MODEL_NAME = "google/gemma-3-270m-it"
DTYPE = torch.float32
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE,).to(device)

print("Model loaded on:", model.device)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
W1024 12:30:14.932000 62722 .venv/lib/python3.14/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Model loaded on: cpu


In [6]:
def inference(model, prompt):
    device = next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=True,
            temperature=0.7
        )

    return inputs, tokenizer.decode(outputs[0], skip_special_tokens=True)

In [7]:
def evaluate_model(model, tokenizer):
    model.eval()
    device = next(model.parameters()).device

    total_loss = 0.0
    total_tokens = 0
    for prompt in val_data:
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=1024,
        ).to(device)

        with torch.no_grad():
            outputs = model(
                **inputs,
                labels=inputs["input_ids"],
                use_cache=False,
            )

        token_count = inputs["input_ids"].shape[1] - 1
        total_loss += outputs.loss.item() * token_count
        total_tokens += token_count

    return { "loss": total_loss / total_tokens }

In [8]:
evaluate_model(model, tokenizer)

{'loss': 3.762071979278721}

In [9]:
def describe_model(model):
    print(type(model).__name__)
    print(model.model.layers)

In [10]:
tokenized_prompt, answer = inference(model, prompt="Answer in one word what is 1 + 1")
print(answer)

Answer in one word what is 1 + 1 = 2?

(a)  The number 1
(b)  The number 2
(c)  The number 3
(d)  The number 4
(e)  The number 5

**Answer:** (b) The number 2



## Model structure

In [11]:
describe_model(model)

Gemma3ForCausalLM
ModuleList(
  (0-17): 18 x Gemma3DecoderLayer(
    (self_attn): Gemma3Attention(
      (q_proj): Linear(in_features=640, out_features=1024, bias=False)
      (k_proj): Linear(in_features=640, out_features=256, bias=False)
      (v_proj): Linear(in_features=640, out_features=256, bias=False)
      (o_proj): Linear(in_features=1024, out_features=640, bias=False)
      (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
      (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
    )
    (mlp): Gemma3MLP(
      (gate_proj): Linear(in_features=640, out_features=2048, bias=False)
      (up_proj): Linear(in_features=640, out_features=2048, bias=False)
      (down_proj): Linear(in_features=2048, out_features=640, bias=False)
      (act_fn): GELUTanh()
    )
    (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_attention_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (pre_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_feedforward_layernorm): Gemma3RMSNorm((640,

In [12]:
print(f"Parameters before pruning: {tp.utils.count_params(model)}")

Parameters before pruning: 268098176


## 1. Smallest MLP channels

Delete MLP channels with the smallest L2 norm. Each channel is removed jointly from gate, up, and down.

In [13]:
from copy import deepcopy
from torch import nn


def forward_fn(model, input_ids):
    return model(input_ids=input_ids, use_cache=False, return_dict=False)[0]


def prune_channels(model, ratio=0.25):
    model = deepcopy(model)

    mlp_roots = {
        proj
        for block in model.model.layers
        for proj in (block.mlp.gate_proj, block.mlp.up_proj, block.mlp.down_proj)
    }

    ignored_layers = [
        module
        for module in model.modules()
        if isinstance(module, nn.Linear) and module not in mlp_roots
    ]

    pruner = tp.pruner.MagnitudePruner(
        model=model,
        example_inputs=tokenized_prompt["input_ids"].to(model.device),
        forward_fn=forward_fn,
        importance=tp.importance.GroupMagnitudeImportance(p=2),
        pruning_ratio=ratio,
        ignored_layers=ignored_layers,
        root_module_types=[nn.Linear],
        round_to=8,
    )

    pruner.step()
    return model

In [14]:
pruned_model = prune_channels(model, ratio=0.1)

/Users/mac/PycharmProjects/small-LLM/.venv/lib/python3.14/site-packages/torch_pruning/dependency/graph.py:390: UserWarning: Unwrapped parameters detected: ['model.layers.16.self_attn.k_norm.weight', 'model.layers.12.post_feedforward_layernorm.weight', 'model.layers.6.post_attention_layernorm.weight', 'model.layers.12.self_attn.q_norm.weight', 'model.layers.1.pre_feedforward_layernorm.weight', 'model.layers.2.self_attn.k_norm.weight', 'model.layers.11.self_attn.k_norm.weight', 'model.layers.3.self_attn.q_norm.weight', 'model.layers.2.self_attn.q_norm.weight', 'model.layers.5.self_attn.k_norm.weight', 'model.layers.5.post_feedforward_layernorm.weight', 'model.layers.12.post_attention_layernorm.weight', 'model.layers.12.self_attn.k_norm.weight', 'model.layers.13.pre_feedforward_layernorm.weight', 'model.layers.9.post_feedforward_layernorm.weight', 'model.layers.9.self_attn.k_norm.weight', 'model.layers.2.post_feedforward_layernorm.weight', 'model.layers.3.post_attention_layernorm.weight',

In [15]:
print(inference(pruned_model, "Hello, how are you?")[1])

Hello, how are you? ***ign

 ME
 mode,
up and
 b c's and sonic-aim
 2- pra, and, 
,
me'un
and b b
 on-un, and 
    ,  for cb, and
b's/for and b-ing  
 'at the  on best,
, and the,
,
so,
im   and 
 & at the 


In [16]:
print(pruned_model.model.layers)

ModuleList(
  (0-17): 18 x Gemma3DecoderLayer(
    (self_attn): Gemma3Attention(
      (q_proj): Linear(in_features=640, out_features=1024, bias=False)
      (k_proj): Linear(in_features=640, out_features=256, bias=False)
      (v_proj): Linear(in_features=640, out_features=256, bias=False)
      (o_proj): Linear(in_features=1024, out_features=640, bias=False)
      (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
      (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
    )
    (mlp): Gemma3MLP(
      (gate_proj): Linear(in_features=640, out_features=1840, bias=False)
      (up_proj): Linear(in_features=640, out_features=1840, bias=False)
      (down_proj): Linear(in_features=1840, out_features=640, bias=False)
      (act_fn): GELUTanh()
    )
    (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_attention_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (pre_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
  )


In [17]:
print(f"Parameters after pruning: {tp.utils.count_params(pruned_model)}")

Parameters after pruning: 260909696


### Loss

In [18]:
print("Original:", evaluate_model(model, tokenizer))
print("Pruned:  ", evaluate_model(pruned_model, tokenizer))

Original: {'loss': 3.762071979278721}
Pruned:   {'loss': 22.69308587813449}


**Result.** Smallest-norm channels are a poor choice. Loss increased about 6x.

## 2. Largest MLP channels

Same MLP cut, but delete the channels with the largest L2 norm.

In [19]:
def prune_model_2(model, ratio=0.1):
    model = deepcopy(model)

    mlp_roots = {
        proj
        for block in model.model.layers
        for proj in (block.mlp.gate_proj, block.mlp.up_proj, block.mlp.down_proj)
    }

    ignored_layers = [
        module
        for module in model.modules()
        if isinstance(module, nn.Linear) and module not in mlp_roots
    ]

    pruner = tp.MagnitudePruner(
        model=model,
        example_inputs=tokenized_prompt["input_ids"].to(model.device),
        forward_fn=forward_fn,
        importance=tp.importance.GroupMagnitudeImportance(p=2, normalizer=lambda s: -s),
        pruning_ratio=ratio,
        ignored_layers=ignored_layers,
        root_module_types=[nn.Linear],
        round_to=8,
    )

    pruner.step()
    return model

In [20]:
pruned_model2 = prune_model_2(model, ratio=0.1)

/Users/mac/PycharmProjects/small-LLM/.venv/lib/python3.14/site-packages/torch_pruning/dependency/graph.py:390: UserWarning: Unwrapped parameters detected: ['model.layers.2.post_feedforward_layernorm.weight', 'model.layers.5.self_attn.q_norm.weight', 'model.layers.11.self_attn.k_norm.weight', 'model.layers.16.self_attn.q_norm.weight', 'model.layers.4.input_layernorm.weight', 'model.layers.5.pre_feedforward_layernorm.weight', 'model.layers.10.post_attention_layernorm.weight', 'model.layers.12.post_feedforward_layernorm.weight', 'model.layers.15.input_layernorm.weight', 'model.layers.16.pre_feedforward_layernorm.weight', 'model.norm.weight', 'model.layers.17.self_attn.k_norm.weight', 'model.layers.12.input_layernorm.weight', 'model.layers.0.pre_feedforward_layernorm.weight', 'model.layers.3.post_attention_layernorm.weight', 'model.layers.4.post_feedforward_layernorm.weight', 'model.layers.11.self_attn.q_norm.weight', 'model.layers.8.self_attn.k_norm.weight', 'model.layers.14.post_attentio

In [21]:
print(inference(pruned_model2, "Hello, how are you?")[1])

Hello, how are you?
Preferred accordingly??.?.?.?.?.?.?.?.?.?.?....??....??..?..?..?..?..?........??....??....??..?..?..?..?..?.....??.?.?.?.?..?..?..?..?..?..…??.?. vẫn??.?. vẫn??.. vẫn??.. vẫn??.. vẫn??.. vẫn??.. vẫn??.. vẫn??.. vẫn??.. vẫn??.. vẫn??.. vẫn??.. vẫn??.. vẫn??.. vẫn??.. vẫn??..???


In [22]:
print(pruned_model2.model.layers)

ModuleList(
  (0-17): 18 x Gemma3DecoderLayer(
    (self_attn): Gemma3Attention(
      (q_proj): Linear(in_features=640, out_features=1024, bias=False)
      (k_proj): Linear(in_features=640, out_features=256, bias=False)
      (v_proj): Linear(in_features=640, out_features=256, bias=False)
      (o_proj): Linear(in_features=1024, out_features=640, bias=False)
      (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
      (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
    )
    (mlp): Gemma3MLP(
      (gate_proj): Linear(in_features=640, out_features=1840, bias=False)
      (up_proj): Linear(in_features=640, out_features=1840, bias=False)
      (down_proj): Linear(in_features=1840, out_features=640, bias=False)
      (act_fn): GELUTanh()
    )
    (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_attention_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (pre_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
  )


In [23]:
print(f"Parameters after pruning: {tp.utils.count_params(pruned_model2)}")

Parameters after pruning: 260909696


### Loss

In [24]:
print("Original:", evaluate_model(model, tokenizer))
print("Pruned:  ", evaluate_model(pruned_model2, tokenizer))

Original: {'loss': 3.762071979278721}
Pruned:   {'loss': 9.054247155563816}


**Result.** Better than the smallest-norm cut, but loss is still about 3x worse.

## 3. Random MLP channels

Delete random MLP channels, jointly from gate, up, and down.

In [25]:
def prune_model3(model, ratio=0.1, seed=42):
    model = deepcopy(model)

    torch.manual_seed(seed)

    for block in model.model.layers:
        n_prune = int(block.mlp.gate_proj.out_features * ratio)
        idx = torch.randperm(block.mlp.gate_proj.out_features)[:n_prune].tolist()

        tp.prune_linear_out_channels(block.mlp.gate_proj, idx)
        tp.prune_linear_out_channels(block.mlp.up_proj, idx)
        tp.prune_linear_in_channels(block.mlp.down_proj, idx)

    return model

In [26]:
pruned_model3 = prune_model3(model)

In [27]:
print(inference(pruned_model3, "Hello, how are you?")[1])

Hello, how are you?
Is there a chance of rain tomorrow evening?
I'm trying to find out what's going on with the news about the new ringgit (Silver Crystal Entacson).
I've just learned about it, but I'm not sure about the details.
What are your thoughts on what will happen if the ringgit falls?
I'm planning on to go out tonight, but I'm not sure if I'll see what happens.
I


In [28]:
print(pruned_model3.model.layers)

ModuleList(
  (0-17): 18 x Gemma3DecoderLayer(
    (self_attn): Gemma3Attention(
      (q_proj): Linear(in_features=640, out_features=1024, bias=False)
      (k_proj): Linear(in_features=640, out_features=256, bias=False)
      (v_proj): Linear(in_features=640, out_features=256, bias=False)
      (o_proj): Linear(in_features=1024, out_features=640, bias=False)
      (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
      (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
    )
    (mlp): Gemma3MLP(
      (gate_proj): Linear(in_features=640, out_features=1844, bias=False)
      (up_proj): Linear(in_features=640, out_features=1844, bias=False)
      (down_proj): Linear(in_features=1844, out_features=640, bias=False)
      (act_fn): GELUTanh()
    )
    (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_attention_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (pre_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
  )


In [29]:
print(f"Parameters after pruning: {tp.utils.count_params(pruned_model3)}")

Parameters after pruning: 261047936


In [30]:
print("Original:", evaluate_model(model, tokenizer))
print("Pruned:  ", evaluate_model(pruned_model3, tokenizer))

Original: {'loss': 3.762071979278721}
Pruned:   {'loss': 6.439094986779418}


**Result.** Best MLP cut so far. Loss rose by almost 2x.

## 4. Attention heads

Remove whole query heads from `q_proj` and the matching inputs of `o_proj`.

In [31]:
def prune_heads(model, heads_num):
    model = deepcopy(model)

    head_size = model.config.head_dim

    delete_range = [i for head in heads_num for i in range(head * head_size, (head + 1) * head_size)]

    for block in model.model.layers:
        attn = block.self_attn
        q_proj, o_proj = attn.q_proj, attn.o_proj
        tp.prune_linear_out_channels(q_proj, delete_range)
        tp.prune_linear_in_channels(o_proj, delete_range)

        model.config.num_attention_heads -= len(heads_num)

    return model

In [32]:
pruned_model4 = prune_heads(model, [0])

In [33]:
print(pruned_model4.model.layers)

ModuleList(
  (0-17): 18 x Gemma3DecoderLayer(
    (self_attn): Gemma3Attention(
      (q_proj): Linear(in_features=640, out_features=768, bias=False)
      (k_proj): Linear(in_features=640, out_features=256, bias=False)
      (v_proj): Linear(in_features=640, out_features=256, bias=False)
      (o_proj): Linear(in_features=768, out_features=640, bias=False)
      (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
      (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
    )
    (mlp): Gemma3MLP(
      (gate_proj): Linear(in_features=640, out_features=2048, bias=False)
      (up_proj): Linear(in_features=640, out_features=2048, bias=False)
      (down_proj): Linear(in_features=2048, out_features=640, bias=False)
      (act_fn): GELUTanh()
    )
    (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_attention_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (pre_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
  )
)


In [34]:
print(f"Parameters after pruning: {tp.utils.count_params(pruned_model4)}")

Parameters after pruning: 262199936


In [35]:
print("Original:", evaluate_model(model, tokenizer))

for head in range(model.config.num_attention_heads):
    new_model = prune_heads(model, [head])
    print(f"Pruned (deleted head: {head}):  ", evaluate_model(new_model, tokenizer))

Original: {'loss': 3.762071979278721}
Pruned (deleted head: 0):   {'loss': 6.9507034381028}
Pruned (deleted head: 1):   {'loss': 10.358075373957778}
Pruned (deleted head: 2):   {'loss': 7.416869957818614}
Pruned (deleted head: 3):   {'loss': 4.964996977196919}


### Heads 0 and 2 together

In [36]:
pruned_model_head02 = prune_heads(model, [0, 2])
print("Pruned (deleted head: [0, 2]):  ", evaluate_model(pruned_model_head02, tokenizer))

Pruned (deleted head: [0, 2]):   {'loss': 13.903848714339134}


**Result.** Two heads hurt much more than one. Keep a single-head cut.

## 5. Whole blocks

The model has 18 decoder blocks. Delete a whole block.

In [37]:
def prune_blocks(model, blocks_num):
    model = deepcopy(model)
    drop = set(blocks_num)

    model.model.layers = nn.ModuleList(block for i, block in enumerate(model.model.layers) if i not in drop)
    model.config.layer_types = [kind for i, kind in enumerate(model.config.layer_types) if i not in drop]
    model.config.num_hidden_layers = len(model.model.layers)

    for new_index, block in enumerate(model.model.layers):
        block.layer_idx = new_index
        block.self_attn.layer_idx = new_index

    return model

In [38]:
pruned_model5 = prune_blocks(model, [0])

In [39]:
print(pruned_model5.model.layers)

ModuleList(
  (0-16): 17 x Gemma3DecoderLayer(
    (self_attn): Gemma3Attention(
      (q_proj): Linear(in_features=640, out_features=1024, bias=False)
      (k_proj): Linear(in_features=640, out_features=256, bias=False)
      (v_proj): Linear(in_features=640, out_features=256, bias=False)
      (o_proj): Linear(in_features=1024, out_features=640, bias=False)
      (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
      (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
    )
    (mlp): Gemma3MLP(
      (gate_proj): Linear(in_features=640, out_features=2048, bias=False)
      (up_proj): Linear(in_features=640, out_features=2048, bias=False)
      (down_proj): Linear(in_features=2048, out_features=640, bias=False)
      (act_fn): GELUTanh()
    )
    (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_attention_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (pre_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
  )


In [40]:
print(f"Parameters after pruning: {tp.utils.count_params(pruned_model5)}")

Parameters after pruning: 262524544


In [41]:
print("Original:", evaluate_model(model, tokenizer))

for block in range(len(model.model.layers)):
    new_model = prune_blocks(model, [block])
    print(f"Pruned (deleted block: {block}):  ", evaluate_model(new_model, tokenizer))

Original: {'loss': 3.762071979278721}
Pruned (deleted block: 0):   {'loss': 13.741168926346347}
Pruned (deleted block: 1):   {'loss': 24.491463463272286}
Pruned (deleted block: 2):   {'loss': 5.552441718188603}
Pruned (deleted block: 3):   {'loss': 13.217021415697515}
Pruned (deleted block: 4):   {'loss': 4.632076993436385}
Pruned (deleted block: 5):   {'loss': 14.033922541834055}
Pruned (deleted block: 6):   {'loss': 4.236766526314408}
Pruned (deleted block: 7):   {'loss': 4.092196664731873}
Pruned (deleted block: 8):   {'loss': 4.1274701213601475}
Pruned (deleted block: 9):   {'loss': 3.956135412754908}
Pruned (deleted block: 10):   {'loss': 5.131008808513201}
Pruned (deleted block: 11):   {'loss': 5.817351503255666}
Pruned (deleted block: 12):   {'loss': 5.795061940326086}
Pruned (deleted block: 13):   {'loss': 6.285704697240472}
Pruned (deleted block: 14):   {'loss': 5.244687169081186}
Pruned (deleted block: 15):   {'loss': 5.065710371413165}
Pruned (deleted block: 16):   {'loss': 

### Several blocks

In [42]:
pruned_model_blocks = prune_blocks(model, [7, 9])
print("Pruned (deleted block: [7, 9]):  ", evaluate_model(pruned_model_blocks, tokenizer))
print(f"Parameters after pruning: {tp.utils.count_params(pruned_model_blocks)}")

Pruned (deleted block: [7, 9]):   {'loss': 4.710071361609793}
Parameters after pruning: 256950912


**Result.** About 11M fewer parameters and loss raised only by 20%. Dropping blocks is worth keeping.